# 6.4. 多输入多输出通道

当我们处理彩色图像时，图像有多个颜色通道（如RGB的3个通道）。本节将介绍如何使用多个输入和输出通道的卷积层。

In [ ]:
import tensorflow as tf
import numpy as np

## 6.4.1. 多输入通道

当输入包含多个通道时，需要构造一个与输入数据具有相同输入通道数的卷积核，以便与输入数据进行互相关运算。

In [ ]:
def corr2d_multi_in(X, K):
    """计算多输入通道的二维互相关运算"""
    # 先遍历X和K的第0个维度（通道维度），再把它们加在一起
    return tf.reduce_sum([d2l_corr2d(x, k) for x, k in zip(X, K)], axis=0)

def d2l_corr2d(X, K):
    """二维互相关运算"""
    h, w = K.shape
    Y = tf.Variable(tf.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1)))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j].assign(tf.reduce_sum(X[i: i + h, j: j + w] * K))
    return Y

In [ ]:
# 验证多输入通道的互相关运算
X = tf.constant([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
                [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = tf.constant([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

print("输入 X 形状 (通道数, 高, 宽):", X.shape)
print("卷积核 K 形状 (通道数, 高, 宽):", K.shape)
print("\n输出:")
print(corr2d_multi_in(X, K).numpy())

## 6.4.2. 多输出通道

在最流行的神经网络架构中，我们会增加通道维度，通常在向上的层级中增加。

In [ ]:
def corr2d_multi_in_out(X, K):
    """计算多输入多输出通道的二维互相关运算"""
    # 迭代K的第0个维度，每次都对输入X执行互相关运算
    # 最后将所有结果都叠加在一起
    return tf.stack([corr2d_multi_in(X, k) for k in K], 0)

In [ ]:
# 构造一个具有3个输出通道的卷积核
K = tf.stack((K, K + 1, K + 2), 0)
print("卷积核 K 形状 (输出通道数, 输入通道数, 高, 宽):", K.shape)

In [ ]:
# 对输入张量X与卷积核张量K执行互相关运算
Y = corr2d_multi_in_out(X, K)
print("输出 Y 形状 (输出通道数, 高, 宽):", Y.shape)
print("\n输出 Y:")
print(Y.numpy())

## 6.4.3. 1×1 卷积层

1×1卷积，即$k_h = k_w = 1$，看起来似乎没有多大意义。毕竟，卷积相关性运算在高度和宽度维度上没有任何操作。但是，1×1卷积的主要作用是调整网络层的通道数。

In [ ]:
def corr2d_multi_in_out_1x1(X, K):
    """1×1卷积的等价实现"""
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = tf.reshape(X, (c_i, h * w))
    K = tf.reshape(K, (c_o, c_i))
    # 全连接层中的矩阵乘法
    Y = tf.matmul(K, X)
    return tf.reshape(Y, (c_o, h, w))

In [ ]:
# 验证1×1卷积
X = tf.random.normal((3, 3, 3), 0, 1)
K = tf.random.normal((2, 3, 1, 1), 0, 1)

Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)

print("两种方法结果是否相同:", float(tf.reduce_sum(tf.abs(Y1 - Y2))) < 1e-6)
print("\n1×1卷积本质上是每个像素位置应用全连接层！")

## 6.4.4. 使用TensorFlow内置卷积层

In [ ]:
# 多通道卷积示例
print("实际应用示例：")

# 输入：批量大小=1, 高=32, 宽=32, 通道=3 (RGB图像)
X = tf.random.uniform((1, 32, 32, 3))

# 3个输入通道 -> 64个输出通道
conv1 = tf.keras.layers.Conv2D(64, kernel_size=3, padding='same')
Y1 = conv1(X)
print(f"第一层: {X.shape} -> {Y1.shape}")

# 64个输入通道 -> 128个输出通道
conv2 = tf.keras.layers.Conv2D(128, kernel_size=3, padding='same')
Y2 = conv2(Y1)
print(f"第二层: {Y1.shape} -> {Y2.shape}")

# 使用1×1卷积调整通道数：128 -> 256
conv3 = tf.keras.layers.Conv2D(256, kernel_size=1)
Y3 = conv3(Y2)
print(f"1×1卷积: {Y2.shape} -> {Y3.shape}")

## 小结

1. **多输入通道**：卷积核对每个输入通道执行互相关运算，并将结果相加
2. **多输出通道**：每个输出通道都有自己的卷积核集合
3. **1×1卷积**：
   - 不识别空间模式，只处理通道
   - 相当于每个像素位置应用全连接层
   - 常用于调整通道数、增加非线性
4. 卷积核形状：(输出通道数, 输入通道数, 高, 宽)